## Reading data into dataframe

In [0]:
from pyspark.sql.functions import col, sum, when ,count,countDistinct

In [0]:
date='2026-10-19'

In [0]:
payers_df=spark.read\
                .option('inferSchema','True')\
                .option('header','True')\
                .csv(f"abfss://bronze@stheathproject.dfs.core.windows.net/payers/{date}/payers.csv")  

payers_df.printSchema()

## Fixing column names and type and handling null values

In [0]:
payers_df=payers_df.withColumnRenamed('Id','payers_id')

payers_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in payers_df.columns
]).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# 1. Select the Bronze Encounters DataFrame
df = payers_df

# 2. Standardize column names
for col_name in df.columns:
    new_name = (
        col_name.strip()
        .lower()
        .replace(" ", "_")
    )
    df = df.withColumnRenamed(col_name, new_name)

# 3. Inspect the transformed schema
df.printSchema()

# 4. Validate row count
print("Silver input rows:", df.count())

# 5. Check primary-key nulls
display(
    df.filter(F.col("payers_id").isNull())
)

# 6. Check duplicate encounter IDs
display(
    df.groupBy("payers_id")
      .count()
      .filter(F.col("count") > 1)
)

## Writing data into external Location 

In [0]:
silver_path = "abfss://silver@stheathproject.dfs.core.windows.net/Payers/"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
silver_df=spark.read\
    .format('delta')\
    .load(silver_path)



print("Bronze/input rows:", df.count())
print("Silver rows:", silver_df.count())

display(silver_df.limit(10))

## Registering table with external Location

In [0]:
%sql
CREATE TABLE IF NOT EXISTS health_care.silver.silver_payers
USING DELTA
LOCATION 'abfss://silver@stheathproject.dfs.core.windows.net/Payers/';